In [1]:
!wget https://data.ncl.ac.uk/ndownloader/articles/24574753/versions/1 -O ../data/Gutenberg.zip
!mkdir ../data/Gutenberg
!unzip ../data/Gutenberg.zip -d ../data/Gutenberg/
!rm ../data/Gutenberg.zip

--2025-05-11 17:42:14--  https://data.ncl.ac.uk/ndownloader/articles/24574753/versions/1
Resolving data.ncl.ac.uk (data.ncl.ac.uk)... 54.216.15.6, 52.48.143.184, 34.255.146.38
Connecting to data.ncl.ac.uk (data.ncl.ac.uk)|54.216.15.6|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 128836454 (123M) [application/zip]
Saving to: ‘../data/Gutenberg.zip’

../data/Gutenberg.z 100%[===================>] 122.87M  40.9MB/s    in 3.0s    

2025-05-11 17:42:18 (40.9 MB/s) - ‘../data/Gutenberg.zip’ saved [128836454/128836454]

Archive:  ../data/Gutenberg.zip
 extracting: ../data/Gutenberg/test_x.npy  
 extracting: ../data/Gutenberg/test_y.npy  
 extracting: ../data/Gutenberg/train_x.npy  
 extracting: ../data/Gutenberg/train_y.npy  
 extracting: ../data/Gutenberg/valid_x.npy  
 extracting: ../data/Gutenberg/valid_y.npy  
 extracting: ../data/Gutenberg/README  
 extracting: ../data/Gutenberg/metadata  


In [ ]:
import torch
from torch import nn
from torch.utils.data import DataLoader, random_split, TensorDataset
from torchvision import datasets, transforms
import os
import random
import numpy as np

SEED = 0
torch.manual_seed(SEED)
os.environ['PYTHONHASHSEED'] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)
g = torch.Generator()
g.manual_seed(0)

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
if torch.cuda.is_available():
    torch.cuda.manual_seed(SEED)
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    torch.use_deterministic_algorithms(True)
    os.environ['CUBLAS_WORKSPACE_CONFIG'] = ':4096:8'  # or ':16:8'
    os.environ['CUDA_LAUNCH_BLOCKING'] = '1'
    torch.set_deterministic_debug_mode(True)



################################
#     RESTART     RUNTIME      #
################################
from ssonn.model.utils import *
from ssonn.metrics.nonlinearity_metrics import *
from ssonn.metrics.edge_finder import *
from ssonn.metrics.train_metrics import *
from ssonn.train.train import *

In [2]:
class SimpleFCN(nn.Module):
    def __init__(self, input_size=27 * 18, output_size=6):
        super(SimpleFCN, self).__init__()
        self.fc0 = nn.Linear(input_size, output_size)

    def forward(self, x):
        return self.fc0(x)

In [3]:
hyperparams = {
    "num_epochs": 64,
    "batch_size": 256,
    "metric": AbsGradientEdgeMetric(nn.CrossEntropyLoss()),
    "aggregation_mode": "mean",
    "choose_thresholds": {"fc0": 0.05},
    "choose_thresholds_del": {"fc0": 0.1},
    "threshold": 0.05,
    "min_delta_epoch_replace": 32,
    "window_size": 5,
    "lr": 5e-4,
    "delete_after": 4,
    "task_type": "classification",
    "fully_connected": False,
    "max_to_replace": 2000, # None -> no limit
    "weight_decay": 1e-3
}

name = ", ".join(
    f"{key}: {value.__class__.__name__ if key == 'metric' else value}"
    for key, value in hyperparams.items()
)

In [4]:
train_x = np.load("../data/Gutenberg/train_x.npy")
train_y = np.load("../data/Gutenberg/train_y.npy")
valid_x = np.load("../data/Gutenberg/valid_x.npy")
valid_y = np.load("../data/Gutenberg/valid_y.npy")
test_x = np.load('../data/Gutenberg/test_x.npy')
test_y = np.load('../data/Gutenberg/test_y.npy')

train_tensor_x = torch.tensor(train_x).float().view(-1, 27 * 18)
train_tensor_y = torch.tensor(train_y).long()

valid_tensor_x = torch.tensor(valid_x).float().view(-1, 27 * 18)
valid_tensor_y = torch.tensor(valid_y).long()

test_tensor_x = torch.tensor(test_x).float().view(-1, 27 * 18)
test_tensor_y = torch.tensor(test_y).long()

train_dataset = TensorDataset(train_tensor_x, train_tensor_y)
valid_dataset = TensorDataset(valid_tensor_x, valid_tensor_y)
test_dataset = TensorDataset(test_tensor_x, test_tensor_y)

train_loader = DataLoader(train_dataset, batch_size=hyperparams['batch_size'], shuffle=True, generator=g)
val_loader = DataLoader(valid_dataset, batch_size=hyperparams['batch_size'], shuffle=False, generator=g)
test_loader = DataLoader(test_dataset, batch_size=hyperparams['batch_size'], shuffle=False, generator=g)

In [5]:
model = SimpleFCN()
sparse_model = convert_dense_to_sparse_network(model, layers=[model.fc0], device=device)

In [6]:
import wandb
wandb.login()
wandb.finish()
run = wandb.init(
    project="self-expanding-nets-Gutenberg",
    name=f"trash",
    config=hyperparams
)

wandb: Currently logged in as: vanyamironov to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


In [7]:
import time
start_time = time.time()

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(sparse_model.parameters(), lr=hyperparams['lr'], weight_decay=hyperparams['weight_decay'])
train_sparse_recursive(sparse_model, train_loader, train_loader, val_loader, criterion, optimizer, hyperparams, device)

print("--- %s seconds ---" % (time.time() - start_time))

100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 176/176 [00:02<00:00, 68.86it/s]


Epoch 1/64, Train Loss: 1.7800, Val Loss: 1.7659, Val Accuracy: 0.2447


100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 176/176 [00:02<00:00, 74.58it/s]


Epoch 2/64, Train Loss: 1.7507, Val Loss: 1.7455, Val Accuracy: 0.2703


100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 176/176 [00:01<00:00, 112.99it/s]


Epoch 3/64, Train Loss: 1.7307, Val Loss: 1.7312, Val Accuracy: 0.2825


... (59 outputs left out)

Epoch 33/64, Train Loss: 1.6553, Val Loss: 1.6799, Val Accuracy: 0.3007


100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 176/176 [00:01<00:00, 115.60it/s]


Epoch 34/64, Train Loss: 1.6550, Val Loss: 1.6799, Val Accuracy: 0.3032
Chosen edges: tensor([[  0,   0,   0,  ...,   5,   5,   5],
        [  0,   1,   2,  ..., 483, 484, 485]], device='cuda:0') 1963


KeyboardInterrupt: 

In [8]:
_, accuracy = eval_one_epoch(sparse_model, criterion, test_loader, hyperparams['task_type'], device)
params = get_params_amount(sparse_model)

In [9]:
accuracy, params

(0.4215, 701941)

In [ ]:
torch.save(sparse_model, 'gutenberg.pt')

In [10]:
import pandas as pd

data = {
    'Model': ['Ours', 'ResNet-18', 'AlexNet', 'VGG16', 'ConvNext', 'MNASNet', 'DenseNet', 'ResNeXt', 'PC-DARTS', 'DrNAS', 'Bonsai-Net', 'DARTS', ' Bonsai', 'Random'],
    'Accuracy (%)': [accuracy * 100, 49.98, 45.53, 44.00, 31.93, 38.00, 43.28, 40.30, 49.12, 46.62, 48.57, 47.72, 29.00, 16.6],
    'Parameters': [params, 11_689_512, 61_100_840, 138_357_544, 88_591_464, 4_383_312, 28_681_000, 25_028_904, None, None, None, None, None, None]
}

table = pd.DataFrame(data)

def format_with_commas(x):
    return "{:,}".format(x)

styled_table = (table.style
               .format({'Accuracy (%)': '{:.2f}',
                       'Parameters': format_with_commas})
               .set_properties(**{'text-align': 'center'})
               .set_table_styles([
                   {'selector': 'th', 'props': [('text-align', 'center')]},
                   {'selector': 'caption', 'props': [('font-size', '1.1em')]}
               ])
               .hide(axis='index'))

styled_table

Model,Accuracy (%),Parameters
Ours,42.15,"701,941.0"
ResNet-18,49.98,"11,689,512.0"
AlexNet,45.53,"61,100,840.0"
VGG16,44.00,"138,357,544.0"
ConvNext,31.93,"88,591,464.0"
MNASNet,38.00,"4,383,312.0"
DenseNet,43.28,"28,681,000.0"
ResNeXt,40.30,"25,028,904.0"
PC-DARTS,49.12,nan
DrNAS,46.62,nan
